# Clase 04 — Manipulación de Datos con Pandas

**Duplicados, `map`/`apply`, GroupBy y Pivot Tables, fechas y resampling — sobre un dataset real de rendimiento de jugadores.**

Antes de los 5 módulos nuevos hacemos un **Módulo 0 de repaso**: comandos de Pandas que ya usamos en la Clase 03 pero que no llegamos a explicar con el detalle que merecen (`.loc`/`.iloc`, filtros booleanos combinados, `value_counts`, `sort_values`, `rename`, `drop`, un `groupby` simple).

## Módulo 0 — Repaso: NumPy Relámpago y Comandos Clásicos de Pandas

### Sobre el dataset

Trabajamos con `fifa_world_cup_2026_player_performance.csv`: **54.600 filas**, cada una es la **aparición de un jugador en un partido** (no un jugador único — si un jugador disputó 5 partidos, aparece 5 veces). 75 columnas, agrupadas en tres familias: datos del jugador (`player_name`, `age`, `nationality`, `position`, `market_value_eur`...), datos del partido (`match_id`, `match_date`, `opponent_team`, `tournament_stage`...) y métricas de rendimiento en ese partido (`goals`, `assists`, `shots`, `pass_accuracy`, `minutes_played`, `player_rating`...).

Son 1.248 jugadores únicos, 48 equipos, 1.050 partidos, sin nulos ni filas duplicadas.

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("fifa_world_cup_2026_player_performance.csv")

print(df.shape)     # (54600, 75)
df.info()

### 0.1 NumPy — repaso relámpago

Una columna de un DataFrame se puede convertir en un `ndarray` de NumPy con `.to_numpy()`. A partir de ahí, todas las operaciones son **vectorizadas**, sin `for`.

In [ ]:
valores_mercado = df["market_value_eur"].to_numpy()   # de Serie de Pandas a ndarray de NumPy
print(type(valores_mercado))

valores_en_millones = valores_mercado / 1_000_000       # vectorizado: sin for
print(f"Promedio: {valores_en_millones.mean():.2f}M | Máximo: {valores_en_millones.max():.2f}M")
print(f"Percentil 90: {np.percentile(valores_en_millones, 90):.2f}M")

### 0.2 Pandas — Series y DataFrame: lo esencial

`df["col"]` (un corchete) devuelve una **Serie**; `df[["col1", "col2"]]` (doble corchete) devuelve un **DataFrame**.

In [ ]:
nombres = df["player_name"]                          # un corchete -> Serie
ficha = df[["player_name", "team", "position"]]       # doble corchete -> DataFrame

print(type(nombres), type(ficha))
display(ficha.head(3))

### 0.3 Selección con `.loc[]` y `.iloc[]`

- **`.iloc[filas, columnas]`**: selección **por posición** (números enteros), sin importar cómo se llamen filas o columnas.
- **`.loc[filas, columnas]`**: selección **por etiqueta**, y se combina naturalmente con un filtro booleano en la parte de "filas".

In [ ]:
primeras_filas = df.iloc[0:5, 0:4]                 # posición: filas 0-4, columnas 0-3
print(primeras_filas)

goleadores = df.loc[df["goals"] > 2, ["player_name", "team", "goals"]]   # etiqueta: filtro + columnas por nombre
print(goleadores)

### 0.4 Filtrado booleano con múltiples condiciones

Sobre Series de Pandas se usan `&` (y), `|` (o) y `~` (no) — nunca `and`/`or`/`not` de Python puro — y cada condición individual va entre paréntesis.

`player_rating == 0` ocurre exactamente en las mismas filas donde `minutes_played == 0`: son jugadores convocados que no llegaron a jugar ese partido.

In [ ]:
jugaron = df[df["minutes_played"] > 0]                                              # excluye a los que no jugaron ese partido
delanteros_caros = df[(df["position"] == "Forward") & (df["market_value_eur"] > 50_000_000)]
no_finales = df[~(df["tournament_stage"] == "Final")]                               # ~ invierte la condición

print(f"Apariciones con minutos jugados: {len(jugaron)}")
print(f"Delanteros con valor > 50M€: {len(delanteros_caros)}")
print(f"Apariciones fuera de la Final: {len(no_finales)}")

### 0.5 Explorar categorías y ordenar: `value_counts()`, `unique()`, `nunique()`, `sort_values()`

- **`value_counts()`**: cuenta cuántas veces aparece cada valor distinto de una columna, de mayor a menor.
- **`nunique()`**: cuántos valores distintos hay (un solo número).
- **`sort_values("columna", ascending=False)`**: reordena todo el DataFrame según una columna.

In [ ]:
print(df["position"].value_counts())
print(df["team"].nunique())        # 48 equipos distintos

top_valor = df.sort_values("market_value_eur", ascending=False).head(5)
display(top_valor[["player_name", "team", "market_value_eur"]])

### 0.6 Preprocesamiento: nulos, `rename()` y `drop()`

`rename()` le pone otro nombre a una columna sin tocar los datos; `drop(columns=[...])` saca columnas completas (para filas se usa `index=[...]`).

In [ ]:
print(df.isnull().sum().sum())    # 0 -> este dataset viene sin nulos

df_renombrado = df.rename(columns={"goals": "goles", "assists": "asistencias"})   # renombrar para trabajar en español
df_renombrado = df_renombrado.drop(columns=["jersey_number"])                      # sacar una columna que no vamos a usar

print(df_renombrado.columns[:5].tolist())

### 0.7 Agregación básica con `groupby`

El caso más simple: una columna para agrupar y una sola métrica. El `groupby` con múltiples agregaciones a la vez y las `pivot_table` se ven más adelante en esta clase.

In [ ]:
valor_por_posicion = df.groupby("position")["market_value_eur"].mean().sort_values(ascending=False)
print((valor_por_posicion / 1_000_000).round(2))

### 0.8 Sinergia NumPy + Pandas

`np.where(condición, si_true, si_false)` es un `if/else` vectorizado, aplicado a toda una columna de una sola vez — sin `for` y sin `.apply()` (que recién se introduce en el próximo módulo).

In [ ]:
df["tuvo_gol"] = np.where(df["goals"] > 0, "Sí", "No")   # if/else vectorizado, sin for ni apply
print(df["tuvo_gol"].value_counts())

## Módulo 1 — Valores Faltantes y Duplicados

### El lenguaje de la ausencia: NaN, None, Null y vacío

`NaN` es el estándar técnico de Pandas/NumPy para faltantes numéricos. `None` es el "vacío" nativo de Python — Pandas lo convierte en `NaN` apenas entra a un DataFrame. Una cadena vacía (`""`) **no** es un nulo, es texto válido.

Un detalle contraintuitivo: `np.nan == np.nan` da `False`. Por eso existen `isna()`/`isnull()` en vez de comparar con `==`.

In [ ]:
print(np.nan == np.nan)   # False -> NaN nunca es igual a sí mismo

data = {"Edad": [25, np.nan, 30], "Ciudad": ["Madrid", "Bogotá", None]}
df_ejemplo = pd.DataFrame(data)
print(df_ejemplo.isna())   # None también se detecta como nulo

### Detección y cuantificación

`fifa_world_cup_2026_player_performance.csv` viene sin nulos ni duplicados, así que simulamos ambos problemas sobre una copia (`df_sucio`) — como si hubieran ocurrido fallas reales de captura.

`info()` da el panorama general por columna. `df.isna().sum()` cuenta nulos por columna (`True` vale 1); dividido por `len(df)` da el porcentaje, la métrica que realmente importa.

In [ ]:
df_sucio = df.copy()

# Simulamos métricas que no se pudieron registrar en vivo (falla de captura real)
df_sucio.loc[50, "player_rating"] = np.nan
df_sucio.loc[120, "pass_accuracy"] = np.nan
df_sucio.loc[120, "distance_covered_km"] = np.nan   # dos nulos en la MISMA fila
df_sucio.loc[300, "nationality"] = np.nan

nulos_totales = df_sucio.isna().sum()
porcentaje_nulos = (nulos_totales / len(df_sucio) * 100).round(4)
print(porcentaje_nulos[porcentaje_nulos > 0])   # solo las columnas afectadas

### `duplicated()` y `drop_duplicates()`

`duplicated()` marca `True` las filas que ya aparecieron antes (compara todos los valores por defecto). `subset=[...]` busca duplicados solo en columnas puntuales — por ejemplo, un ID que debería ser único. `drop_duplicates(keep="first")` limpia el DataFrame, conservando la primera aparición.

Simulamos el error típico de un sistema que carga el mismo registro dos veces.

In [ ]:
# Simulamos que el sistema cargó por duplicado las apariciones de las filas 100 a 102
df_sucio = pd.concat([df_sucio, df_sucio.iloc[100:103]], ignore_index=True)

print(f"Duplicados exactos: {df_sucio.duplicated().sum()}")

df_limpio = df_sucio.drop_duplicates(keep="first")
print(f"Filas antes: {len(df_sucio)} | Filas después: {len(df_limpio)}")

### Criterios de limpieza: ¿eliminar o imputar?

**Eliminar (`dropna()`)**: válido cuando sobran datos y perder un 2% no afecta estadísticamente, o cuando falta justo la columna "etiqueta" que se quiere predecir. **Imputar**: media (distribución normal, sin outliers), mediana (más robusta ante extremos), moda (categóricas), o un valor constante cuando conviene conservar la fila sin inventar un número.

In [ ]:
media_rating = df_sucio["player_rating"].mean()
df_sucio["player_rating"] = df_sucio["player_rating"].fillna(media_rating)   # numérica -> media

moda_nacionalidad = df_sucio["nationality"].mode()[0]
df_sucio["nationality"] = df_sucio["nationality"].fillna(moda_nacionalidad)   # categórica -> moda

df_sucio = df_sucio.dropna(subset=["pass_accuracy", "distance_covered_km"])   # sin sustituto razonable -> eliminar

print(df_sucio.isna().sum().sum())   # 0 -> ya no quedan nulos

### El dilema de los duplicados: no todos se borran

No todo duplicado es un error: **identidad** (misma persona, mismo producto, mismo segundo exacto) es error del sistema → se borra; **eventos** (mismo cliente comprando lo mismo dos días seguidos) son dos hechos reales → se conservan. Acá cada fila ya tiene `match_id` + `player_id`: dos filas con esa misma combinación **sí** son un error, porque un jugador no puede tener dos apariciones distintas en el mismo partido.

**Mapa mental del módulo**: explorá (`info()`, `isna().sum()`) → contextualizá (¿error o realidad del negocio?) → medí en porcentajes, no en absolutos → decidí (`dropna()`, imputar, o descartar la columna) → deduplicá según las claves de negocio que correspondan.

## Módulo 2 — Transformaciones con `map` y `apply`

### El método `map()`: la tabla de traducción

`.map()` se usa sobre una **Series** y es ideal cuando hay una correspondencia clara, uno a uno, como un diccionario de traducción. Si un valor no está como clave en el diccionario, `map()` lo convierte en `NaN` — hay que cubrir todos los casos.

In [ ]:
diccionario_pie = {"Left": "Izquierdo", "Right": "Derecho"}

df["pie_habil"] = df["preferred_foot"].map(diccionario_pie)
print(df["pie_habil"].value_counts())

### El método `apply()`: flexibilidad total

Sobre una Series transforma con una función (no solo un diccionario). Con `axis=1` sobre un DataFrame, le pasa a la función la **fila completa** — ahí sirve para reglas de negocio que combinan varias columnas.

In [ ]:
df["equipo_mayuscula"] = df["team"].apply(lambda x: x.upper())   # transformación simple con lambda

def calcular_riesgo(fila):
    if fila["fouls_committed"] >= 2 and fila["yellow_cards"] == 1:
        return "Riesgo disciplinario"
    return "Bajo"

df["riesgo"] = df.apply(calcular_riesgo, axis=1)   # axis=1 -> la función recibe la FILA completa
print(df["riesgo"].value_counts())

**Diferencias clave**: `map` (Series, sustitución simple con diccionario) vs. `apply` (Series o DataFrame, funciones complejas o lógica entre columnas con `axis=1`) vs. `applymap` (DataFrame completo, una función a todas las celdas a la vez).

**Recomendaciones de oro**: priorizar la vectorización (si se puede con `df["A"] + df["B"]`, no usar `apply`), validar después con `value_counts()`, y cuidado con los `NaN` — pueden romper una función personalizada.

## Módulo 3 — Agrupar, Resumir y Comparar: GroupBy y Pivot Tables

### Agregaciones múltiples con `agg()`

`.agg([...])` recibe una lista de funciones y las aplica todas a la vez — útil cuando hace falta el total **y** el promedio para comparar volumen contra rendimiento.

In [ ]:
resumen_goles = df.groupby("position")["goals"].agg(["sum", "mean", "count"])
print(resumen_goles.round(2))

### Agrupación por múltiples columnas y Pivot Tables

Pasar una **lista** de columnas al `groupby` crea una jerarquía. Una `pivot_table` muestra esa misma idea como una matriz 2D (`index` en filas, `columns` en columnas, `values` + `aggfunc` en las celdas) — más fácil de leer de un vistazo.

In [ ]:
tabla_rating = df.pivot_table(
    index="tournament_stage", columns="position", values="player_rating", aggfunc="mean"
).round(2)
display(tabla_rating)

Los arqueros tienen un rating promedio de ~2.0–2.1 en todas las instancias, muy por debajo del resto (~3.7–4.0): no es que jueguen peor, es que el 66,7% de las apariciones de arquero son de suplentes que no llegaron a jugar (rating en 0, el hallazgo del Módulo 0), contra ~39% en las demás posiciones — cada plantel lleva varios arqueros, pero solo uno juega por partido.

**Errores comunes**: confundir `count()` (cuántas filas) con `sum()` (cuánto suman); Pandas ignora los nulos por defecto en `mean()`/`sum()`; y la columna agrupada pasa a ser el índice del resultado — `.reset_index()` la devuelve a columna normal.

## Módulo 4 — Fechas, Series Temporales y Resampling

### Conversión a Datetime y el índice temporal

Al cargar el CSV, `match_date` es texto. `pd.to_datetime()` la convierte en fechas reales. Convertirla en el **índice** del DataFrame habilita acceder por período (`.loc["2026-06"]`) y es el requisito para el `resample()`.

In [ ]:
df["match_date"] = pd.to_datetime(df["match_date"])
print(df["match_date"].dtype)   # datetime64[...], ya no object

df_temporal = df.set_index("match_date").sort_index()   # sort_index: fechas en orden antes de operar

junio = df_temporal.loc["2026-06"]
print(f"Apariciones jugador-partido en junio: {len(junio)}")

### Resampling: cambiando el "zoom" de tus datos

`resample()` es un `groupby` especializado en tiempo. Downsampling de partido-por-partido a semana-por-semana: contamos partidos únicos y sumamos goles.

In [ ]:
partidos_por_semana = (
    df_temporal.drop_duplicates("match_id")   # cada partido cuenta una sola vez, no una por jugador
    .resample("W")
    .size()
)
print(partidos_por_semana.head())

goles_por_semana = df_temporal.resample("W")["goals"].sum()
print(goles_por_semana.head())

**Errores comunes**: no ordenar el índice antes de remuestrear (`.sort_index()` es obligatorio); operar con fechas todavía en texto (por eso `to_datetime()` va primero); y confundir la pregunta de negocio — total de la semana (`.sum()`) o promedio por partido (`.mean()`) responden cosas distintas.

## Módulo 5 — Manipulación de Datos: Pandas (Síntesis)

### Combinación de fuentes: Merge vs. Concat

`merge` es como el `JOIN` de SQL: busca una columna en común y fusiona las tablas lateralmente (agrega columnas). `concat` apila una tabla debajo de otra (agrega filas). Probamos ambas: un `merge` con una tabla de referencia a propósito incompleta, y un `concat` que reparte el dataset en dos mitades cronológicas y las vuelve a unir.

In [ ]:
# Tabla de referencia chica, A PROPÓSITO incompleta (solo 10 de los 48 equipos)
df_confederaciones = pd.DataFrame({
    "team": ["Argentina", "Brazil", "France", "Spain", "Germany",
             "Japan", "Morocco", "Mexico", "Nigeria", "Qatar"],
    "confederacion": ["CONMEBOL", "CONMEBOL", "UEFA", "UEFA", "UEFA",
                       "AFC", "CAF", "CONCACAF", "CAF", "AFC"],
})

df_con_confederacion = pd.merge(df, df_confederaciones, on="team", how="left")
print(df_con_confederacion["confederacion"].isna().sum())   # equipos sin match en la tabla chica

# concat: partimos el dataset en dos mitades cronológicas y las volvemos a unir
primera_mitad = df[df["match_date"] < "2026-07-01"]
segunda_mitad = df[df["match_date"] >= "2026-07-01"]
df_reunido = pd.concat([primera_mitad, segunda_mitad])
print(len(df_reunido) == len(df))   # True -> concat no perdió ni duplicó filas

**Errores comunes**: el índice no es una columna normal, es el sistema de direcciones de las filas (pensarlo al hacer `.reset_index()`); filtrar puede dar una "ventana" a la tabla original en vez de una copia independiente (`.copy()` explícito antes de modificar); y al sumar dos Series, Pandas alinea por etiqueta de índice, no por posición física.

In [ ]:
# Patrón seguro: .copy() explícito antes de modificar un recorte
delanteros = df[df["position"] == "Forward"].copy()
delanteros["goles_por_90"] = delanteros["goals"] / (delanteros["minutes_played"] / 90)

---

## Pre-Entrega: Estructura Inicial del Dataset del Proyecto (Checkpoint 1)

Con NumPy y Pandas ya recorridos, este checkpoint es el momento de dejar de trabajar con datasets de ejemplo y empezar a darle forma a tu propio Proyecto Final. Lo que construyas hoy no es el análisis definitivo — es la base saneada y explorada sobre la que en el próximo módulo vas a aplicar visualización.

**Qué construir**: un informe de inspección inicial en un notebook, con la carga y el "diagnóstico" de tu dataset:

| Bloque | Qué incluye |
|---|---|
| **Carga del Dataset** | `read_csv` (o similar) para traer tus datos al entorno. |
| **Estructura Base** | Reporte de filas y columnas (`.shape`). |
| **Diagnóstico Estructural** | `.info()` para identificar nulos y `dtypes` que no coincidan con la realidad. |
| **Resumen Numérico** | `.describe()` para entender los rangos de tus variables. |
| **Saneamiento Inicial** | Al menos 3 filtros o selecciones booleanas + eliminar al menos una columna. |

**Errores comunes**: ignorar los `dtypes` (una columna numérica leída como texto no se puede sumar), y no guardar los cambios (`df[df["edad"] > 18]` no modifica `df` a menos que lo reasignes).

Ejemplo completo abajo, sobre el dataset de esta clase.

### Carga e Inspección

In [ ]:
df = pd.read_csv("fifa_world_cup_2026_player_performance.csv")
display(df.head())

print(f"Dimensiones: {df.shape}")   # (54600, 75)
df.info()

Revisando `dtypes`: `market_value_eur` es `int64` (correcto), `match_date` es `object` porque todavía no la convertimos a fecha (eso pasa en el Módulo 4) — nada aparece como texto por error, a diferencia de un dataset real donde un símbolo de moneda puede "ensuciar" una columna numérica.

### Perfilado Inicial

In [ ]:
nulos_por_columna = df.isnull().sum()
print(f"Total de valores nulos: {nulos_por_columna.sum()}")

display(df.describe())

Mirando la tabla real: `age` va de 17 a 39 años (rango de jugadores de un Mundial); `market_value_eur` va de ~529 mil a 200 millones de euros, con una media (~20M) muy por debajo del máximo — señal de que unas pocas estrellas empujan el promedio hacia arriba.

### Saneamiento y Selección

In [ ]:
jugaron = df[df["minutes_played"] > 0]                                              # filtro 1: excluye suplentes que no jugaron
delanteros_caros = df[(df["position"] == "Forward") & (df["market_value_eur"] > 50_000_000)]  # filtro 2: condición combinada
fase_eliminatoria = df[df["tournament_stage"] != "Group Stage"]                     # filtro 3: todo lo que no es fase de grupos

df_reducido = df.drop(columns=["jersey_number"])   # eliminamos una columna que no aporta al análisis

print(f"Apariciones con minutos jugados: {len(jugaron)}")
print(f"Delanteros con valor > 50M€: {len(delanteros_caros)}")
print(f"Apariciones en fase eliminatoria: {len(fase_eliminatoria)}")

### Reflexión

**¿Qué problemas encontré en los datos?** Ninguno grave: el dataset viene con los `dtypes` correctos y sin valores nulos — la única transformación pendiente es convertir `match_date` de texto a fecha real, que se resuelve en el Módulo 4. El mayor punto de atención no es de calidad de datos sino de interpretación: cada fila es una *aparición* jugador-partido, no un jugador único, así que cualquier conteo tiene que aclarar sobre qué unidad está parado.

**¿Cuáles van a ser mis variables clave?** `player_rating` y `market_value_eur` para medir rendimiento y valor; `position` y `tournament_stage` para segmentar comparaciones; `minutes_played` como filtro obligatorio antes de calcular cualquier promedio de rendimiento, para no mezclar titulares con suplentes que no jugaron.

### Exportación

*File → Download as → PDF via HTML* (o imprimir la página como PDF). Nombre del archivo: `Apellido_Nombre_Checkpoint1.pdf`.